# 02 Compact core citation graph

Scan edges twice to build a shared CSR index. Deduplicate references within citing papers, remove self-links and invalid dates, and retain same-year citations and historical sources.

In [ ]:
# Select the project Python kernel before running this cell.
from pathlib import Path
import os, sys, json
candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT = next((p for p in candidates if (p / 'assembly' / 'runtime.py').is_file()), None)
if PROJECT is None:
    raise FileNotFoundError('Open this notebook from the repository root or a subdirectory.')
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
CONFIG_PATH = Path(os.environ.get('GKA_CONFIG', str(PROJECT / 'config.json'))).expanduser().resolve()
settings = json.loads(CONFIG_PATH.read_text(encoding='utf-8-sig'))
os.environ['ASSEMBLY_THREADS'] = str(settings['threads'])
import bootstrap
bootstrap.configure(settings['threads'])
from assembly.runtime import config, run, dirs
C = config(CONFIG_PATH)
D = dirs(C)
print('Project:', PROJECT)
print('Python:', sys.executable)
print('Output:', D['root'])


In [ ]:
run(C, 'references')

In [ ]:
summary = json.loads((D['status'] / 'references.json').read_text(encoding='utf-8'))['result']
print(json.dumps(summary, indent=2))

ref_ptr, ref_count and ref_index form one aligned index. Keep these caches together. The sorted lookup option reduces hash-index memory use.